# Исследование надёжности заёмщиков

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели **кредитного скоринга** — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

Описание данных:

    children — количество детей в семье
    days_employed — общий трудовой стаж в днях
    dob_years — возраст клиента в годах
    education — уровень образования клиента
    education_id — идентификатор уровня образования
    family_status — семейное положение
    family_status_id — идентификатор семейного положения
    gender — пол клиента
    income_type — тип занятости
    debt — имел ли задолженность по возврату кредитов
    total_income — ежемесячный доход
    purpose — цель получения кредита

## Шаг 1. Откройте файл с данными и изучите общую информацию

In [1]:
import pandas as pd #импорт библиотеки
df = pd.read_csv('/datasets/data.csv') #читаем файл и сохраняем его в переменной df.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
children            21525 non-null int64
days_employed       19351 non-null float64
dob_years           21525 non-null int64
education           21525 non-null object
education_id        21525 non-null int64
family_status       21525 non-null object
family_status_id    21525 non-null int64
gender              21525 non-null object
income_type         21525 non-null object
debt                21525 non-null int64
total_income        19351 non-null float64
purpose             21525 non-null object
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


In [2]:
df.head(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Вывод**

Количество значений в столбцах различается. Это говорит о том, что в данных есть пустые значения. Меньше строк имеенно в столбцах, где хранятся значения о трудовом стаже в днях и ежемесячном доходе, пропуски появились случайно, скорее всего люди просто нигде не работали или не указали свои данные в анкете. В дальнейшем мы заполним эти значения на основе группировки данных. 

А также эти два столбца имеют вещественный тип данных, чего не должно быть.

В следующем пункте мы уже отредактируем таблицу, так как мы уже заметили, что столбец days_employed содержит отрицательные значения, чего не должно быть.

## Шаг 2. Предобработка данных

### Обработка пропусков

Посмотрим первые 10 строк 

In [3]:
df.head(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


Заметим,что в столбце days_employed содержутся отрицательные значения, чего не должно быть. Значит заменим отрицательные значения на модуль числа

In [4]:
df['days_employed'] = df['days_employed'].abs()

Проверим уникальные значения в столбцах

In [5]:
df['dob_years'].value_counts()

35    617
40    609
41    607
34    603
38    598
42    597
33    581
39    573
31    560
36    555
44    547
29    545
30    540
48    538
37    537
50    514
43    513
32    510
49    508
28    503
45    497
27    493
56    487
52    484
47    480
54    479
46    475
58    461
57    460
53    459
51    448
59    444
55    443
26    408
60    377
25    357
61    355
62    352
63    269
64    265
24    264
23    254
65    194
66    183
22    183
67    167
21    111
0     101
68     99
69     85
70     65
71     58
20     51
72     33
19     14
73      8
74      6
75      1
Name: dob_years, dtype: int64

0 лет  - невозможное значение, таких строк меньше 1%, поэтому удаляем эти строки

In [6]:
df = df[df['dob_years']!=0]

In [7]:
df['days_employed'].value_counts()

986.927316       1
386155.404320    1
1645.463049      1
4236.274243      1
6620.396473      1
                ..
5619.328204      1
448.829898       1
1687.038672      1
2348.524271      1
582.538413       1
Name: days_employed, Length: 19260, dtype: int64

Максимальный возраст в таблице - 75 лет, даже если человек работает с 14 лет, то его стаж работы в днях не должен быть больше 365*61, но такие значения в таблице есть. Нам важен этот столбец для анализа, поэтому найдем отношение медианы таких работников и медианы нормальных работников, а затем на этот коэфицент разделим неподходящие значения

In [8]:
k = df[df['days_employed']>365*61]['days_employed'].median()/df[df['days_employed']<=365*61]['days_employed'].median()

In [9]:
df.loc[df['days_employed']>365*61,'days_employed'] = df[df['days_employed']>365*61]['days_employed'] / k

In [10]:
df['children'].value_counts()

 0     14080
 1      4802
 2      2042
 3       328
 20       75
-1        47
 4        41
 5         9
Name: children, dtype: int64

Не может быть -1 ребенок, эти 47 значений заменяем на модуль числа. А также не понятно откуда появилось значение 20, скорее всего был приписан лишний 0.

In [11]:
df['children'] = df['children'].abs()
df.loc[df['children']==20,'children'] = 2

Проверим данные на наличие пропусков вызовом методов для суммирования пропущенных значений

In [12]:
df.isnull().sum()

children               0
days_employed       2164
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2164
purpose                0
dtype: int64

Чтобы заполнить пропуски в ежемесячном доходе - сгруппируем данные по education_id и найдем средние значение методом agg()

In [13]:
df.groupby('education_id')['total_income'].agg(['mean'])

,mean
education_id,
0,207327.677426
1,153752.210093
2,181502.449011
3,132155.513626
4,174750.155792


Заполняем пропуски методом fillna()

In [14]:
df.loc[(df['education_id']==0),'total_income']=df['total_income'].fillna(207327.677426)

In [15]:
df.loc[(df['education_id']==1),'total_income']=df['total_income'].fillna(153752.210093)

In [16]:
df.loc[(df['education_id']==2),'total_income']=df['total_income'].fillna(181502.449011)

In [17]:
df.loc[(df['education_id']==3),'total_income']=df['total_income'].fillna(132155.513626)

In [18]:
df.loc[(df['education_id']==4),'total_income']=df['total_income'].fillna(174750.155792)

Пропуски в стаже заполняем медианным значением 

In [19]:
median_days_employed=df['days_employed'].median()
df['days_employed']=df['days_employed'].fillna(median_days_employed)

Проверяем данные на наличие пропусков 

In [20]:
df.isnull().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

**Вывод**

Мы отредактировали таблицу, ошибки появились скорее всего из-за человеческого фактора. А точнее мы избавились от отрицательных значений, от невозможных значений(возраст, количество детей, стаж работы в днях, кол-во детей)
А также мы заполнили пропуски медианным значением и на основне группировки в столбцах days_employed и total_income, что поможет провести анализ точнее. Пропуски в данных столбцах появились скорее всего случайно: из-за  того что человек ни разу не работал или просто не стал указывать об этом в анкете.

### Замена типа данных

Заменяем вещественный тип данных в стобцах days_employed и total_income

In [21]:
df['days_employed']=df['days_employed'].astype('int')
df['total_income']=df['total_income'].astype('int')

Проверяем информацию о таблице

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 21424 entries, 0 to 21524
Data columns (total 12 columns):
children            21424 non-null int64
days_employed       21424 non-null int64
dob_years           21424 non-null int64
education           21424 non-null object
education_id        21424 non-null int64
family_status       21424 non-null object
family_status_id    21424 non-null int64
gender              21424 non-null object
income_type         21424 non-null object
debt                21424 non-null int64
total_income        21424 non-null int64
purpose             21424 non-null object
dtypes: int64(7), object(5)
memory usage: 2.7+ MB


**Вывод**

Мы заменили вещественный тип данных на целочисленный методом astype(). Метод to_numeric() не подходит, потому что превращает в вещественный тип данных 

### Обработка дубликатов

Для начала приведем строки в колонках к нижнему регистру 

In [23]:
df['education'] = df['education'].str.lower()
df['family_status'] = df['family_status'].str.lower()
df['income_type'] = df['income_type'].str.lower()
df['purpose'] = df['purpose'].str.lower()

Необходимо установить наличие дубликатов

In [24]:
df.duplicated().sum()#получение суммарного количества дубликатов в таблице

71

In [25]:
df = df.drop_duplicates().reset_index(drop=True)#удаление всех дубликатов из таблицы 

Проверка на отсутствие дубликатов

In [26]:
df.duplicated().sum()

0

**Вывод**

Дубликаты появились случайным образом. Например: сбой данных, анкету человек заполнил несколько раз и тд. Таких строк меньше 1% и их удаление не повлияет на результат. Удаление дубликатов позволит провести анализ точнее.

### Лемматизация

Импортируем библиотеку

In [27]:
from pymystem3 import Mystem
m = Mystem() 

Напишем функцию сначала для одной строки (которая лемматизирует столбец 'purpose' в строке), а потом применим ко всему столбцу df через метод apply()

In [28]:
def to_lemmas(row):
    text=row['purpose']
    lemmas = m.lemmatize(text)
    return lemmas

In [29]:
 df['lemmas'] = df.apply(to_lemmas, axis=1)

Проверим

In [30]:
df['lemmas'].value_counts()

[автомобиль, \n]                                          966
[свадьба, \n]                                             786
[на,  , проведение,  , свадьба, \n]                       764
[сыграть,  , свадьба, \n]                                 760
[операция,  , с,  , недвижимость, \n]                     672
[покупка,  , коммерческий,  , недвижимость, \n]           658
[покупка,  , жилье,  , для,  , сдача, \n]                 649
[операция,  , с,  , коммерческий,  , недвижимость, \n]    648
[операция,  , с,  , жилье, \n]                            646
[покупка,  , жилье, \n]                                   640
[жилье, \n]                                               640
[покупка,  , жилье,  , для,  , семья, \n]                 637
[строительство,  , собственный,  , недвижимость, \n]      633
[недвижимость, \n]                                        629
[операция,  , со,  , свой,  , недвижимость, \n]           627
[строительство,  , жилой,  , недвижимость, \n]            621
[покупка

**Вывод**

Мы нашли леммы столбца целей кредита, в дальнейшем нам будет легче поделить цели на категории.

### Категоризация данных

Проведем классификацию по уровню зарплаты методом qcut(на 3 категории).

In [31]:
df['total_income_category']=pd.qcut(df['total_income'],3)

Проведем классификацию по типу цели кредита. Напишем функцию для одной строки, а затем применим ко всему столбцу методом .apply()

In [32]:
def purpose_group(purpose):
    if 'жилье' in purpose or 'недвижимость' in purpose:
        return'на жилье'
    if 'свадьба' in purpose:
        return'на свадьбу'
    if 'автомобиль' in purpose:
        return'на автомобиль'
    if 'образование' in purpose:
        return'на образование'

In [33]:
df['purpose_category']=df['lemmas'].apply(purpose_group)

Проверим, посмотрев первые 10 строк

In [34]:
df.head(10)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,lemmas,total_income_category,purpose_category
0,1,8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,"[покупка, , жилье, \n]","(178126.0, 2265604.0]",на жилье
1,1,4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,"[приобретение, , автомобиль, \n]","(20666.999, 122864.667]",на автомобиль
2,0,5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,"[покупка, , жилье, \n]","(122864.667, 178126.0]",на жилье
3,3,4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,"[дополнительный, , образование, \n]","(178126.0, 2265604.0]",на образование
4,0,1519,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,"[сыграть, , свадьба, \n]","(122864.667, 178126.0]",на свадьбу
5,0,926,27,высшее,0,гражданский брак,1,M,компаньон,0,255763,покупка жилья,"[покупка, , жилье, \n]","(178126.0, 2265604.0]",на жилье
6,0,2879,43,высшее,0,женат / замужем,0,F,компаньон,0,240525,операции с жильем,"[операция, , с, , жилье, \n]","(178126.0, 2265604.0]",на жилье
7,0,152,50,среднее,1,женат / замужем,0,M,сотрудник,0,135823,образование,"[образование, \n]","(122864.667, 178126.0]",на образование
8,2,6929,35,высшее,0,гражданский брак,1,F,сотрудник,0,95856,на проведение свадьбы,"[на, , проведение, , свадьба, \n]","(20666.999, 122864.667]",на свадьбу
9,0,2188,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425,покупка жилья для семьи,"[покупка, , жилье, , для, , семья, \n]","(122864.667, 178126.0]",на жилье


А теперь выведем словари. Посмотрим значения в столбцах.

education и education_id:

In [35]:
df['education'].value_counts()

среднее                15108
высшее                  5215
неоконченное высшее      742
начальное                282
ученая степень             6
Name: education, dtype: int64

In [36]:
df['education_id'].value_counts()

1    15108
0     5215
2      742
3      282
4        6
Name: education_id, dtype: int64

family_status и family_status_id:

In [37]:
df['family_status'].value_counts()

женат / замужем          12290
гражданский брак          4130
не женат / не замужем     2794
в разводе                 1185
вдовец / вдова             954
Name: family_status, dtype: int64

In [38]:
df['family_status_id'].value_counts()

0    12290
1     4130
4     2794
3     1185
2      954
Name: family_status_id, dtype: int64

Образование(на основе столбцов education и education_id): высшее  0;
среднее 1;
неоконченное высшее 2;
начальное 3;
ученая степень 4.

Семейный статус(на основе столбцов family_status и family_status_id): женат / замужем 0, гражданский брак 1, вдовец / вдова 2, в разводе 3, не женат / не замужем 4

**Вывод**

Мы категоризировали данные из нескольких столбцов для наглядности и для удобства дальнейшего анализа. А так же выделили словари(образование и семейный статус)

## Шаг 3. Ответьте на вопросы

Группируем таблицу по нужной нам категории и к столбцу debt применяем метод agg(), находим среднее значение

- Есть ли зависимость между наличием детей и возвратом кредита в срок?

In [39]:
df.groupby('children')['debt'].agg(['count','mean'])

,count,mean
children,,
0,14022,0.075453
1,4839,0.091341
2,2114,0.095553
3,328,0.082317
4,41,0.097561
5,9,0.000000


**Вывод**

Люди с одним ребенком, двумя и четырьмя детьми просрачивают кредиты чаще, чем люди без детей и люди с тремя детьми. А люди с 5 детьми не просрачивают кредиты вообще.

- Есть ли зависимость между семейным положением и возвратом кредита в срок?

Применим метод сводных таблиц. Индексы - семейный статус, а к значением debt для каждой группы применим функцию mean, чтобы найти долю должников.

In [40]:
df.pivot_table(index='family_status',values='debt',aggfunc=['count','mean'])

,count,mean
,debt,debt
family_status,,
в разводе,1185,0.071730
вдовец / вдова,954,0.064990
гражданский брак,4130,0.093462
женат / замужем,12290,0.075427
не женат / не замужем,2794,0.097709


**Вывод**

Люди которые находятся в гражданском браке или не женаты просрачивают кредиты чаще, чем все остальные(в разводе, женат / замужем, вдовец / вдова), причем вдовы и вдовцы просрачивают кредиты реже всех.

- Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [41]:
df.groupby('total_income_category')['debt'].agg(['count','mean'])

,count,mean
total_income_category,,
"(20666.999, 122864.667]",7118,0.081203
"(122864.667, 178126.0]",7117,0.090066
"(178126.0, 2265604.0]",7118,0.072211


**Вывод**

Люди со "средним заработком"(122864.667-178126.0) просрачивают кредиты чаще, чем люди с "низким заработком"(20666.999-122864.667). А те в свою очередь просрачивают кредиты чаще, чем люди с "высоким заработком"(178126.0-2265604.0)

- Как разные цели кредита влияют на его возврат в срок?

In [42]:
df.groupby('purpose_category')['debt'].agg(['count','mean'])

,count,mean
purpose_category,,
на автомобиль,4284,0.093371
на жилье,10764,0.072371
на образование,3995,0.092616
на свадьбу,2310,0.079654


**Вывод**

Если люди берут кредит на автомобиль или на образование, то они чаще просрачивают кредит, чем люди, которые берут кредит на жилье или на свадьбу.

## Шаг 4. Общий вывод

Мы отредактировали таблицу, убрав невозможные значения в столбцах dob_years(возраст клиента в годах), total_income(ежемесячный доход), days_employed(общий трудовой стаж в днях), children(количество детей в семье).

Заполнили пропуски специальным методом fillna(): в "стаже работы в днях" медианным значением, а в "доходе" на основе группировки по столбцу образования.

Изменили вещественный тип данных на целочисленный в столбцах days_employed и total_income методом astype()

А так же мы удалили дубликаты методами drop_duplicates() и reset_index(). Все выше перечисленное позволило провести более точный анализ.

Мы провели лемматизацию и категоризировали данные, что в дальнейшем позволило наглядно ответить на вопросы.

И наконец мы поняли, что:

1)Люди с одним ребенком, двумя и четырьмя детьми просрачивают кредиты чаще, чем люди без детей и люди с тремя детьми. А люди с 5 детьми не просрачивают кредиты вообще.

2)Люди которые находятся в гражданском браке или не женаты просрачивают кредиты чаще, чем все остальные(в разводе, женат / замужем, вдовец / вдова), причем вдовы и вдовцы просрачивают кредиты реже всех.

3)Люди со "средним заработком"(122864.667-178126.0) просрачивают кредиты чаще, чем люди с "низким заработком"(20666.999-122864.667). А те в свою очередь просрачивают кредиты чаще, чем люди с "высоким заработком"(178126.0-2265604.0)

4)Если люди берут кредит на автомобиль или на образование, то они чаще просрачивают кредит, чем люди, которые берут кредит на жилье или на свадьбу.